In [0]:
%sql
CREATE OR REPLACE TEMP VIEW numbers AS
SELECT *
FROM VALUES
(3),
(4),
(5),
(7),
(8),
(9),
(11)

AS numbers(num);

In [0]:
%sql
select * from numbers

In [0]:
%sql
with CTE_1 as (
    select num, row_number() over (order by num) as rn from numbers
)
select num, rn, num-rn as group from cte_1

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW user_logins AS
SELECT *
FROM VALUES
(1, '2026-06-01'),
(1, '2026-06-02'),
(1, '2026-06-03'),
(1, '2026-06-06'),
(1, '2026-06-07')

AS user_logins(user_id, login_date);

In [0]:
%sql
with cte as (
select user_id, login_date, row_number() over (order by login_date) as rn from user_logins
)
select user_id, min(login_date) as start_date,max(login_date) as end_date, count(*) as days from cte
group by date_sub(login_date, rn) ,user_id

In [0]:
%sql
with cte as (
select user_id, login_date, row_number() over (order by login_date) as rn from user_logins
)
select user_id, login_date, date_sub(login_date, rn) as grp from cte

# Question: User Login Streaks Based on Business Days 

A product team wants to measure user engagement streaks. Unlike calendar-day streaks, weekends (Saturday and Sunday) should not break a streak. 

A user is considered to have logged in on consecutive business days if: 

The next login is on the very next weekday.  

Friday → Monday counts as consecutive.  

Saturday and Sunday are ignored.  

Any missing weekday (Monday–Friday) breaks the streak. 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW user_logins AS
SELECT *
FROM VALUES
(1, '2026-06-01'),
(1, '2026-06-02'),
(1, '2026-06-03'),
(1, '2026-06-05'),
(1, '2026-06-08'),
(1, '2026-06-09'),
(1, '2026-06-11'),

(2, '2026-06-04'),
(2, '2026-06-05'),
(2, '2026-06-08'),
(2, '2026-06-10'),
(2, '2026-06-11')

AS user_logins(user_id, login_date);

In [0]:
%sql
select * from  user_logins

In [0]:
%sql
WITH login_flags AS (

SELECT
    user_id,
    login_date,

    CASE
        WHEN prev_login IS NULL THEN 1

        WHEN DATEDIFF(login_date, prev_login) = 1
             THEN 0

        WHEN DAYOFWEEK(prev_login) = 6
             AND DATEDIFF(login_date, prev_login) = 3
             THEN 0

        ELSE 1
    END AS new_streak

FROM (

    SELECT *,
           LAG(login_date) OVER (
               PARTITION BY user_id
               ORDER BY login_date
           ) AS prev_login
    FROM user_logins

) t
) select * from login_flags

In [0]:
%sql
with prev_login as (
 
select user_id, login_date, lag(login_date) over (partition by user_id order by login_date) as prev_val from user_logins

),

login_streak_flag as(
select 
user_id,
login_date,
case when prev_val is NULL then 1
when date_diff(login_date, prev_val) = 1 then 0
when dayofweek(prev_val) = 6 and date_diff(login_date, prev_val) = 3 then 0
else 1 end as flag
from prev_login
),
login_streak as (
    select user_id, login_date, sum(flag) over (partition by user_id order by login_date) as streak_grp
    from login_streak_flag
)

select user_id, min(login_date) as strt, max(login_date) as end, count(*) as num_days from login_streak
group by user_id, streak_grp
order by user_id

In [0]:
%sql
WITH cte AS (

SELECT
    *,
    LAG(login_date) OVER (
        PARTITION BY user_id
        ORDER BY login_date
    ) AS prev_login
FROM user_logins

),

flags AS (

SELECT
    user_id,
    login_date,

    CASE
        WHEN prev_login IS NULL THEN 1

        WHEN DATEDIFF(login_date, prev_login) = 1
            THEN 0

        WHEN DAYOFWEEK(prev_login) = 6
             AND DATEDIFF(login_date, prev_login) = 3
            THEN 0

        ELSE 1
    END AS new_streak

FROM cte

),

groups AS (

SELECT
    *,
    SUM(new_streak)
    OVER (
        PARTITION BY user_id
        ORDER BY login_date
    ) AS streak_id

FROM flags

)

SELECT
    user_id,
    MIN(login_date) AS streak_start,
    MAX(login_date) AS streak_end,
    COUNT(*) AS business_days_in_streak
FROM groups
GROUP BY user_id, streak_id
ORDER BY user_id, streak_start;

#  Merge overlapping maintenance windows, then find gaps
# 
A platform records planned maintenance windows. Any windows that overlap or touch should be treated as one continuous outage.

Rules:

If start_date <= previous_end_date + 1 day, merge them
Return the merged outage windows
Also return the next merged window start and the gap in days to that next window

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW system_intervals AS
SELECT *
FROM VALUES
('A', '2026-06-01', '2026-06-03'),
('A', '2026-06-03', '2026-06-04'),
('A', '2026-06-08', '2026-06-10'),
('A', '2026-06-09', '2026-06-12'),
('A', '2026-06-15', '2026-06-15'),

('B', '2026-06-02', '2026-06-02'),
('B', '2026-06-05', '2026-06-06'),
('B', '2026-06-07', '2026-06-09')

AS system_intervals(system_id, start_date, end_date);

### Below have edge case missing

In [0]:
%sql
with lag_cte as (
    select system_id, start_date, end_date, lag(end_date) over (partition by system_id order by start_date) as prev_end from system_intervals
),
streak_cte as (
    select 
    system_id, start_date, end_date, sum(streak_flag) over(partition by system_id order by start_date rows between unbounded preceding and current row) as streak_group from (
select 
system_id, start_date, end_date, 
case when prev_end is null then 1
when date_diff (start_date, prev_end) <=0 then 0
else 1 end as streak_flag
from lag_cte
) )
select *,  lead(start_date) over (partition by system_id order by start_date) as next_window  from  (
select system_id, min(start_date) as start_date, max(end_date) as end_date, date_diff(max(end_date), 
min (start_date)) as num_days from streak_cte
group by system_id, streak_group )

In [0]:
%sql
WITH running_end AS (

SELECT
    system_id,
    start_date,
    end_date,

    MAX(end_date) OVER (
        PARTITION BY system_id
        ORDER BY start_date
        ROWS BETWEEN UNBOUNDED PRECEDING
             AND 1 PRECEDING
    ) AS running_max_end

FROM system_intervals

)
select * from running_end

In [0]:
%sql
WITH running_end AS (

SELECT
    system_id,
    start_date,
    end_date,

    MAX(end_date) OVER (
        PARTITION BY system_id
        ORDER BY start_date
        ROWS BETWEEN UNBOUNDED PRECEDING
             AND 1 PRECEDING
    ) AS running_max_end

FROM system_intervals

),

flags AS (

SELECT
    *,

    CASE
        WHEN running_max_end IS NULL THEN 1

        WHEN DATEDIFF(start_date, running_max_end) <= 1
            THEN 0

        ELSE 1
    END AS new_group

FROM running_end

),

groups AS (

SELECT
    *,
    SUM(new_group) OVER (
        PARTITION BY system_id
        ORDER BY start_date
        ROWS BETWEEN UNBOUNDED PRECEDING
             AND CURRENT ROW
    ) AS grp

FROM flags

),

merged AS (

SELECT
    system_id,
    MIN(start_date) AS merged_start,
    MAX(end_date) AS merged_end
FROM groups
GROUP BY system_id, grp

)

SELECT
    system_id,
    merged_start,
    merged_end,

    LEAD(merged_start) OVER (
        PARTITION BY system_id
        ORDER BY merged_start
    ) AS next_merged_start,

    CASE
        WHEN LEAD(merged_start) OVER (
            PARTITION BY system_id
            ORDER BY merged_start
        ) IS NULL
        THEN NULL

        ELSE DATEDIFF(
                LEAD(merged_start) OVER (
                    PARTITION BY system_id
                    ORDER BY merged_start
                ),
                merged_end
             ) - 1
    END AS gap_days_to_next

FROM merged
ORDER BY system_id, merged_start;

### Edge Case

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW system_intervals AS
SELECT *
FROM VALUES
('A', '2026-06-01', '2026-06-10'),
('A', '2026-06-02', '2026-06-03'),
('A', '2026-06-09', '2026-06-12'),
('A', '2026-06-15', '2026-06-18')

AS system_intervals(system_id, start_date, end_date);

In [0]:
%sql
with lag_cte as (
    select system_id, start_date, end_date,prev_end, streak_flag, sum(streak_flag) over (partition by system_id order by start_date rows between unbounded preceding and current row) as streak_grp from (
    select system_id, start_date, end_date,prev_end,
    case when date_diff(start_date, prev_end)<=1 then 0
    else 1 end as streak_flag from
    (
    select system_id, start_date, end_date, lag(end_date) over (partition by system_id order by start_date) as prev_end from system_intervals)
    )
),
 max_cte as (
    select system_id, start_date, end_date,prev_end, streak_flag, sum(streak_flag) over (partition by system_id order by start_date rows between unbounded preceding and current row) as streak_grp from (
    select system_id, start_date, end_date,prev_end,
    case when date_diff(start_date, prev_end)<=1 then 0
    else 1 end as streak_flag from
    (
        select system_id, start_date, end_date, max(end_date) over (partition by system_id order by start_date rows between unbounded preceding and 1 preceding) as prev_end from system_intervals
    )
    )
)
select 'lag', * from lag_cte
union all
select 'max', * from max_cte

### # A warehouse tracks stock movements for items. Each transaction changes the running stock level.
### # 
Rules:

Compute the running stock per item ordered by date
A “positive stock island” starts when stock becomes > 0 after being <= 0
It ends when stock becomes <= 0
Return one row per positive stock island
Also return the next island start and the gap in days to the next island

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW inventory_transactions AS
SELECT *
FROM VALUES
('A', '2026-06-01', 10),
('A', '2026-06-02', -4),
('A', '2026-06-03', -6),
('A', '2026-06-04', 5),
('A', '2026-06-05', -2),
('A', '2026-06-06', -10),
('A', '2026-06-08', 8),
('A', '2026-06-09', 2),
('A', '2026-06-10', -5),
('A', '2026-06-11', 4),

('B', '2026-06-01', 6),
('B', '2026-06-02', -1),
('B', '2026-06-04', -4),
('B', '2026-06-05', -2),
('B', '2026-06-07', 3),
('B', '2026-06-08', -1),
('B', '2026-06-09', -2)

AS inventory_transactions(item_id, txn_date, qty_change);

In [0]:
%sql
with running_sum as (
select item_id, txn_date, qty_change, sum(qty_change) over (partition by item_id order by txn_date rows between unbounded preceding and current row) as running_sum
 from inventory_transactions
),
prev_running_sum as (
select item_id, txn_date, qty_change, sum(qty_change) over (partition by item_id order by txn_date rows between unbounded preceding and current row) as running_sum,
lag(running_sum) over (partition by item_id order by txn_date) as prev_running_sum
 from running_sum
),
streak_flag as (
select item_id, txn_date, qty_change, running_sum , prev_running_sum ,
case when running_sum>0 and prev_running_sum>0 then 0
else 1 end as streak_flag
from prev_running_sum

),
streak_group as (
select item_id, txn_date, qty_change, running_sum , prev_running_sum ,streak_flag, sum(streak_flag) over (partition by item_id order by txn_date rows between unbounded preceding and current row) as streak_group from streak_flag
where running_sum>0
)
select * , lead(start_date) over (partition by item_id order by start_date) as next_window, (date_diff(lead(start_date) over (partition by item_id order by start_date), end_date)-1) as gaps from 
(
select item_id, min(txn_date) as start_date, max(txn_date) as end_date, max(running_sum) as start_stock, min(running_sum) as end_stock  from streak_group
group by item_id, streak_group
)